# 🔄 07: Slowly Changing Dimensions (SCD Types 1, 2, and 3)

In data warehousing, attributes change over time (e.g., customer loyalty tiers, credit limits, addresses).
This notebook demonstrates and compares the **three standard Kimball Slowly Changing Dimension (SCD) strategies**:

| SCD Strategy | Storage Mechanism | History Preserved | Primary Use Case |
| :--- | :--- | :--- | :--- |
| **SCD Type 1** | In-place column overwrite | **Zero history** (latest only) | Correcting typos, non-audit attributes |
| **SCD Type 2** | Historical row versioning | **Full audit trail** (all versions) | Regulatory compliance, point-in-time joins |
| **SCD Type 3** | Adjacent state columns | **Current + Prior state** | Before/after metric comparisons |

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import duckdb

# Add repo root to import medallion functions
curr_dir = Path.cwd().resolve()
if curr_dir.name == 'notebooks':
    repo_root = curr_dir.parent
elif (curr_dir / 'tech-news-arr-warehouse').exists():
    repo_root = curr_dir / 'tech-news-arr-warehouse'
else:
    repo_root = curr_dir
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from part2_slowly_changing_dimensions.scd_pipeline import (
    get_ecommerce_paths,
    load_raw_customers,
    load_customer_spend,
    clean_customers,
    build_scd1_dimension,
    build_scd2_dimension,
    build_scd3_dimension,
    point_in_time_query,
    run_scd_pipeline
)

# 1. Ingest Raw & Clean Silver
paths = get_ecommerce_paths(str(repo_root))
df_raw_cust = load_raw_customers(paths['raw_dir'])
spend_map = load_customer_spend(paths['raw_dir'])
df_silver_cust = clean_customers(df_raw_cust)
print(f"[+] Ingested {len(df_raw_cust)} raw customer records.")
print(f"[+] Calculated spend profiles for {len(spend_map)} active purchasing customers.")
display(df_silver_cust[['customerID', 'companyName', 'contactName', 'city', 'country']].head())


[+] Ingested 91 raw customer records.
[+] Calculated spend profiles for 89 active purchasing customers.
  customerID                         companyName         contactName         city  country
0      ALFKI                 Alfreds Futterkiste        Maria Anders       Berlin  Germany
1      ANATR  Ana Trujillo Emparedados y helados        Ana Trujillo  México D.F.   Mexico
2      ANTON             Antonio Moreno Taquería      Antonio Moreno  México D.F.   Mexico
3      AROUT                     Around the Horn        Thomas Hardy       London       UK
4      BERGS                  Berglunds snabbköp  Christina Berglund        Luleå   Sweden


## 1. SCD Type 1: In-Place Overwrite (Zero History)

- **Mechanism**: Updates the row directly in-place whenever an attribute changes.
- **Grain**: Exactly 1 row per customer (91 rows).
- **Limitation**: Any historical analysis as-of past dates becomes impossible because prior states are destroyed.

In [2]:
df_scd1 = build_scd1_dimension(df_silver_cust, spend_map)

print(f"Total SCD Type 1 Rows: {len(df_scd1)} (Unique Customers: {df_scd1['customer_id'].nunique()})")
print("\n=== Customer BERGS in SCD Type 1 (Only Current Platinum Tier Exists) ===")
display(df_scd1[df_scd1['customer_id'] == 'BERGS'])


Total SCD Type 1 Rows: 91 (Unique Customers: 91)

=== Customer BERGS in SCD Type 1 (Only Current Platinum Tier Exists) ===
  customer_id        company_name        contact_name   city country loyalty_tier  credit_limit_usd last_updated_at
4       BERGS  Berglunds snabbköp  Christina Berglund  Luleå  Sweden     Platinum             50000      1998-04-01


## 2. SCD Type 2: Historical Row Versioning (Full Audit Trail)

- **Mechanism**: Every change creates a **new row** with a synthetic surrogate key (`customer_sk`).
- **Temporal Tracking**: `effective_start_date` and `effective_end_date` define the exact chronological window when this version was valid.
- **Active Indicator**: Current active version has `is_current_flag = True` and `effective_end_date = '9999-12-31'`.
- **Grain**: 1 row per version per customer.

In [3]:
df_scd2 = build_scd2_dimension(df_silver_cust, spend_map)

print(f"Total SCD Type 2 Rows: {len(df_scd2)} across {df_scd2['customer_id'].nunique()} unique customers")
print(f"Active Current Rows:    {df_scd2['is_current_flag'].sum()}")
print(f"Historical Closed Rows: {(~df_scd2['is_current_flag']).sum()}")

print("\n=== Customer BERGS Full Historical Version Audit Trail ===")
display(df_scd2[df_scd2['customer_id'] == 'BERGS'][[
    'customer_sk', 'customer_id', 'company_name', 'loyalty_tier', 
    'credit_limit_usd', 'effective_start_date', 'effective_end_date', 'is_current_flag'
]])


Total SCD Type 2 Rows: 175 across 91 unique customers
Active Current Rows:    91
Historical Closed Rows: 84

=== Customer BERGS Full Historical Version Audit Trail ===
    customer_sk customer_id        company_name loyalty_tier  credit_limit_usd effective_start_date effective_end_date  is_current_flag
6  CUST_SK_0007       BERGS  Berglunds snabbköp       Silver             10000           1996-01-01         1997-06-30            False
7  CUST_SK_0008       BERGS  Berglunds snabbköp         Gold             25000           1997-07-01         1998-03-31            False
8  CUST_SK_0009       BERGS  Berglunds snabbköp     Platinum             50000           1998-04-01         9999-12-31             True


## 3. Point-in-Time Temporal Queries on SCD Type 2

Using SCD Type 2, analysts can answer: **"What was customer BERGS's tier when they placed an order on 1996-08-15?"**

Condition: `effective_start_date <= order_date AND order_date <= effective_end_date`

In [4]:
# Querying across 3 distinct historical milestones
test_dates = ["1996-08-15", "1997-10-01", "1998-05-01"]

print("=== Point-in-Time Lookups for Customer BERGS ===")
for dt in test_dates:
    rec = point_in_time_query(df_scd2, "BERGS", dt)
    print(f"As of Date: {dt} -> Tier: {rec['loyalty_tier']:<10} | Credit Limit: ${rec['credit_limit_usd']:,} | Active Now: {rec['is_current_flag']}")

# SQL Temporal Join with DuckDB
con = duckdb.connect()
con.register('dim_scd2', df_scd2)

query_sql = """
    SELECT 
        customer_sk,
        customer_id,
        loyalty_tier,
        credit_limit_usd,
        effective_start_date,
        effective_end_date,
        is_current_flag
    FROM dim_scd2
    WHERE customer_id = 'BERGS'
      AND '1997-10-01' BETWEEN effective_start_date AND effective_end_date;
"""
print("\n=== DuckDB Point-in-Time SQL Result (As-Of 1997-10-01) ===")
print(con.execute(query_sql).df().to_string(index=False))


=== Point-in-Time Lookups for Customer BERGS ===
As of Date: 1996-08-15 -> Tier: Silver     | Credit Limit: $10,000 | Active Now: False
As of Date: 1997-10-01 -> Tier: Gold       | Credit Limit: $25,000 | Active Now: False
As of Date: 1998-05-01 -> Tier: Platinum   | Credit Limit: $50,000 | Active Now: True

=== DuckDB Point-in-Time SQL Result (As-Of 1997-10-01) ===
 customer_sk customer_id loyalty_tier  credit_limit_usd effective_start_date effective_end_date  is_current_flag
CUST_SK_0008       BERGS         Gold             25000           1997-07-01         1998-03-31            False


## 4. SCD Type 3: Current vs Previous State Columns (Partial History)

- **Mechanism**: Stores the current state AND the immediately previous state in adjacent columns of the same row.
- **Grain**: Exactly 1 row per customer (91 rows).
- **Trade-off**: Zero row explosion, allows fast before/after comparisons, but cannot track 3 or more historical changes.

In [5]:
df_scd3 = build_scd3_dimension(df_silver_cust, spend_map)

print(f"Total SCD Type 3 Rows: {len(df_scd3)} (Unique Customers: {df_scd3['customer_id'].nunique()})")
print("\n=== Customer BERGS in SCD Type 3 (Current vs Previous Columns) ===")
display(df_scd3[df_scd3['customer_id'] == 'BERGS'][[
    'customer_id', 'company_name', 'current_loyalty_tier', 'previous_loyalty_tier',
    'current_credit_limit_usd', 'previous_credit_limit_usd', 'tier_change_date'
]])


Total SCD Type 3 Rows: 91 (Unique Customers: 91)

=== Customer BERGS in SCD Type 3 (Current vs Previous Columns) ===
  customer_id        company_name current_loyalty_tier previous_loyalty_tier  current_credit_limit_usd  previous_credit_limit_usd tier_change_date
4       BERGS  Berglunds snabbköp             Platinum                  Gold                     50000                    25000.0       1998-04-01


## 5. Architectural Comparison Matrix

| Dimension Strategy | Row Count Impact | Audit Trail Fidelity | As-Of Date Joins | Best Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **SCD Type 1** | Constant (N rows) | None (Destructive) | ❌ Not possible | Typo corrections, status updates |
| **SCD Type 2** | Variable (N + M versions) | **100% Complete** | ✅ Fully supported via `BETWEEN` | Audits, regulatory compliance, billing history |
| **SCD Type 3** | Constant (N rows) | Limited (1 prior state) | ❌ Only 2 states available | Before/after campaign lift, immediate upgrades |